# BuildGuard AI — Cloud GPU Model Training

This notebook trains the **BuildGuard AI FloorPlan U-Net** on the **CubiCasa5K** dataset using a free or pro Google Colab NVIDIA GPU (T4 / V100 / A100).

### Pre-requisites:
1. In Google Colab, select: **Runtime** → **Change runtime type** → **Hardware accelerator: GPU (T4)**.
2. Upload the two ZIP files from your local `colab_package/` folder into your Google Drive:
   - `MyDrive/BuildGuardAI/training/BuildGuardAI_training.zip`
   - `MyDrive/BuildGuardAI/dataset/CubiCasa5K_processed.zip`

### Step 1: Verify NVIDIA GPU & CUDA

In [ ]:
import torch

print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device Name:", torch.cuda.get_device_name(0))
    vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"Total GPU Memory: {vram:.2f} GB")
    !nvidia-smi
else:
    raise RuntimeError("GPU is not enabled! Go to Runtime -> Change runtime type -> Select GPU.")

### Step 2: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Ensure the Drive checkpoint folder exists
!mkdir -p /content/drive/MyDrive/BuildGuardAI/checkpoints
print("Google Drive mounted successfully.")

### Step 3: Install Required Python Dependencies

In [ ]:
%pip install -q opencv-python-headless shapely networkx tqdm scikit-learn pillow
print("Dependencies installed.")

### Step 4: Extract Code & Dataset to Local High-Speed Colab Disk (/content)

In [ ]:
import os
import zipfile

PROJECT_DIR = "/content/BuildGuardAI"
DATASET_DIR = "/content/floorplan_dataset"

os.makedirs(PROJECT_DIR, exist_ok=True)
os.makedirs(DATASET_DIR, exist_ok=True)

code_zip = "/content/drive/MyDrive/BuildGuardAI/training/BuildGuardAI_training.zip"
data_zip = "/content/drive/MyDrive/BuildGuardAI/dataset/CubiCasa5K_processed.zip"

assert os.path.isfile(code_zip), f"Could not find code ZIP at {code_zip}. Please check your Google Drive path!"
assert os.path.isfile(data_zip), f"Could not find dataset ZIP at {data_zip}. Please check your Google Drive path!"

print("Extracting training code...")
with zipfile.ZipFile(code_zip, "r") as z:
    z.extractall(PROJECT_DIR)

print("Extracting CubiCasa5K dataset to high-speed NVMe...")
with zipfile.ZipFile(data_zip, "r") as z:
    z.extractall(DATASET_DIR)

print("Extraction complete!")
!ls -la /content/floorplan_dataset/splits

### Step 5: Launch GPU Training with Mixed Precision (FP16)

Checkpoints and history will be saved directly into your Google Drive at:
`MyDrive/BuildGuardAI/checkpoints/`

- If you get disconnected, simply rerun this cell with `--resume` to pick up from the last epoch automatically!

In [ ]:
import os
%cd /content/BuildGuardAI

# Set output directory directly to mounted Google Drive for safe persistence
os.environ["MODEL_OUTPUT_DIR"] = "/content/drive/MyDrive/BuildGuardAI/checkpoints"

!python scripts/train_floorplan_model.py \
    --prepared-dir /content/floorplan_dataset \
    --split-dir /content/floorplan_dataset/splits \
    --checkpoint /content/drive/MyDrive/BuildGuardAI/checkpoints/floorplan_unet_gpu_candidate.pt \
    --epochs 20 \
    --batch-size 8 \
    --learning-rate 0.001 \
    --early-stopping-patience 4 \
    --num-workers 2

### Step 6: Evaluate on the Held-Out Test Set (749 Samples)

In [ ]:
%cd /content/BuildGuardAI

!python scripts/evaluate_floorplan_model.py \
    --prepared-dir /content/floorplan_dataset \
    --split-dir /content/floorplan_dataset/splits \
    --checkpoint /content/drive/MyDrive/BuildGuardAI/checkpoints/floorplan_unet_gpu_candidate.pt \
    --output /content/drive/MyDrive/BuildGuardAI/checkpoints/gpu_test_metrics.json \
    --batch-size 8 \
    --overlay-count 8

print("Evaluation complete! The results are saved to your Google Drive.")